# Formal Supplementary Replot (Celltype Corrected)

Rebuild the requested supplementary figures for chicken heart using:
- the formal chicken-heart training result
- the corrected celltype classifier outputs
- the daily interpolation slices from `formal_daily_piecewise_interpolation_celltypecorrected`
- the visual style of `notebooks/heart` and `downstream_helpers`

In [ ]:
from __future__ import annotations

import io
import json
import logging
import os
import sys
from pathlib import Path

from IPython.display import SVG, display
import anndata as ad
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.patches import FancyArrowPatch
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import seaborn as sns
import torch
from contextlib import redirect_stderr
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score

import os
REPO_ROOT = Path(os.environ.get("CYTOBRIDGE_SOURCE_DIR", ".")).resolve()
PROJECT_DIR = Path(os.environ.get("CYTOBRIDGE_PROJECT_DIR", ".")).resolve()
DATA_DIR = PROJECT_DIR / "data" / "chicken_heart"
sys.path.insert(0, str(REPO_ROOT / "reproduction" / "chicken_heart"))
ANALYSIS_ROOT = Path(os.environ.get("CYTOBRIDGE_HEART_OUTPUT_DIR", PROJECT_DIR / "outputs" / "chicken_heart_paper")).resolve()
HELPER_ROOT = REPO_ROOT / "reproduction" / "chicken_heart" / "downstream_helpers"
DATA_HEART_DIR = DATA_DIR / "raw"
CELLTYPE_SHARE_ROOT = DATA_DIR
import CytoBridge as cb
PACKAGE_ROOT = Path(cb.__file__).resolve().parents[1]

for path in (REPO_ROOT, HELPER_ROOT, PACKAGE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import CytoBridge as cb
from downstream_helpers.heart import HEART_LABEL_TO_COLOR, _prepare_g_heatmap_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sns.set_theme(style="white")
scv.settings.set_figure_params("scvelo")
plt.rcParams["font.family"] = "DejaVu Sans"
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

INTERP_RUN_DIR = ANALYSIS_ROOT / "new_runs_formal_trained" / "formal_daily_piecewise_interpolation_celltypecorrected"
SLICE_DIR = INTERP_RUN_DIR / "slice_data"
MANIFEST_PATH = INTERP_RUN_DIR / "manifest.json"

ALIGNED_H5AD_PATH = DATA_DIR / "aligned.h5ad"
ORIGINAL_H5AD_PATH = DATA_HEART_DIR / "heart_pp.h5ad"
MERGED_WITH_META_PATH = DATA_HEART_DIR / "chicken_heart_spatial_merged_with_meta.h5ad"
MODEL_DIR = DATA_DIR / "model"
EDGE_PREDICTOR_PATH = DATA_DIR / "edge_classifier" / "chicken_heart_edge_model.pt"
OUTPUT_DIR = ANALYSIS_ROOT / "new_runs_formal_trained" / "formal_supplementary_replot_celltypecorrected"
GROWTH_DIR = OUTPUT_DIR / "growth"
ALIGN_DIR = OUTPUT_DIR / "alignment"
VELOCITY_DIR = OUTPUT_DIR / "velocity_components"

for path in (OUTPUT_DIR, GROWTH_DIR, ALIGN_DIR, VELOCITY_DIR):
    path.mkdir(parents=True, exist_ok=True)

with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

LEGACY_REGION_COLOR_MAP = {
    "Ventricle": "#7f7f7f",
    "Outflow tract": "#66c2a5",
    "Endothelium": "#9edae5",
    "Valves": "#c25bac",
    "Atria": "#cab2d6",
    "Epicardium": "#dbdb8d",
    "Trabecular LV and endocardium": "#4575b4",
    "Compact LV and inter-ventricular septum": "#aec7e8",
    "Right ventricle": "#c49c94",
}

MODERN_TIME_COLORS = ["#2B67D5", "#18A3A8", "#EF8250", "#FFC32B"]

print(f"Using device: {DEVICE}")
print(f"Interpolation run dir: {INTERP_RUN_DIR}")
print(f"Formal model dir: {MODEL_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
def canonical_label(value: object) -> str:
    return str(value).replace("\n", " ").replace("\r", " ").replace("  ", " ").strip()


def load_daily_slice_dict(manifest_dict: dict, slice_dir: Path, manifest_key: str = "slice_h5ad") -> tuple[dict[str, ad.AnnData], dict[str, float]]:
    adata_dict: dict[str, ad.AnnData] = {}
    label_to_model_time: dict[str, float] = {}
    for item in sorted(manifest_dict["slices"], key=lambda row: float(row["time_float"])):
        time_label = str(item["time_label"])
        adata_path = Path(item[manifest_key])
        if not adata_path.is_absolute():
            adata_path = slice_dir / adata_path.name
        adata_t = ad.read_h5ad(adata_path)
        if "spatial" not in adata_t.obsm:
            adata_t.obsm["spatial"] = np.asarray(adata_t.X[:, :2], dtype=np.float32)
        adata_t.obs["celltype_prediction"] = adata_t.obs["celltype_prediction"].astype(str).map(canonical_label).values
        adata_t.obs["timepoint"] = time_label
        adata_t.obs["time_label"] = time_label
        adata_t.obs["time_float"] = float(item["time_float"])
        adata_t.obs["is_observed"] = bool(item["is_observed"])
        adata_t.uns["is_observed"] = bool(item["is_observed"])
        adata_dict[time_label] = adata_t
        label_to_model_time[time_label] = float(item["time_float"])
    return adata_dict, label_to_model_time


def attach_region_to_observed_slices(adata_dict: dict[str, ad.AnnData], aligned_h5ad_path: Path) -> None:
    adata_ref = ad.read_h5ad(aligned_h5ad_path)
    region_col = None
    for candidate in ("region", "Region", "anatomic_region"):
        if candidate in adata_ref.obs.columns:
            region_col = candidate
            break
    if region_col is None:
        raise KeyError(f"Aligned h5ad missing region column. Found: {list(adata_ref.obs.columns)}")
    region_lookup = adata_ref.obs[region_col].astype(str).map(canonical_label)
    for time_label, adata_t in adata_dict.items():
        if not bool(adata_t.uns.get("is_observed", False)):
            continue
        shared = adata_t.obs_names.intersection(region_lookup.index)
        series = pd.Series(index=adata_t.obs_names, dtype=object)
        series.loc[shared] = region_lookup.loc[shared].values
        adata_t.obs["region"] = series.fillna("Unknown").astype(str).map(canonical_label).values


def load_celltype_palette(adata_dict: dict[str, ad.AnnData]) -> dict[str, str]:
    palette = {canonical_label(k): str(v) for k, v in HEART_LABEL_TO_COLOR.items()}
    labels = sorted({canonical_label(v) for adata_t in adata_dict.values() for v in adata_t.obs["celltype_prediction"].astype(str)})
    missing = [label for label in labels if label not in palette]
    if missing:
        extra = sns.color_palette("husl", len(missing)).as_hex()
        for label, color in zip(missing, extra):
            palette[label] = mcolors.to_hex(color)
    return {label: palette[label] for label in labels}


def build_region_palette(adata_dict: dict[str, ad.AnnData]) -> dict[str, str]:
    palette = {canonical_label(k): str(v) for k, v in LEGACY_REGION_COLOR_MAP.items()}
    regions = sorted({canonical_label(v) for adata_t in adata_dict.values() if "region" in adata_t.obs.columns for v in adata_t.obs["region"].astype(str)})
    missing = [region for region in regions if region not in palette]
    if missing:
        extra = sns.color_palette("husl", len(missing)).as_hex()
        for region, color in zip(missing, extra):
            palette[region] = mcolors.to_hex(color)
    return {region: palette[region] for region in regions}


daily_adata_dict, label_to_model_time = load_daily_slice_dict(manifest, SLICE_DIR, "slice_h5ad")
attach_region_to_observed_slices(daily_adata_dict, ALIGNED_H5AD_PATH)
time_keys = list(daily_adata_dict.keys())
observed_time_keys = [key for key in time_keys if bool(daily_adata_dict[key].uns.get("is_observed", False))]

label_to_color = load_celltype_palette(daily_adata_dict)
region_to_color = build_region_palette(daily_adata_dict)

loaded = cb.tl.load_dynamical_model_from_dir(
    MODEL_DIR,
    dim=int(daily_adata_dict[time_keys[0]].X.shape[1]),
    device=DEVICE,
    edge_predictor_path=EDGE_PREDICTOR_PATH,
)
model = loaded.model.to(DEVICE)
interaction_threshold = float(getattr(getattr(model, "interaction_net", None), "cutoff", 1000.0))

print("Time keys:", time_keys)
print("Observed time keys:", observed_time_keys)
print("Celltype labels:", sorted(label_to_color))
print("Region labels:", sorted(region_to_color))
print("Loaded formal stages:", loaded.weight_stage, loaded.score_stage)

In [ ]:
def build_daily_feature_dataframe(
    adata_dict: dict[str, ad.AnnData],
    label_to_model_time_map: dict[str, float],
) -> pd.DataFrame:
    rows = []
    feature_dim = int(next(iter(adata_dict.values())).X.shape[1])
    feature_cols = [f"x{i}" for i in range(1, feature_dim + 1)]
    for time_label, adata_t in adata_dict.items():
        X = np.asarray(adata_t.X, dtype=np.float32)
        obs = adata_t.obs.copy()
        dataset_kind = "observed" if bool(adata_t.uns.get("is_observed", False)) else "interpolated"
        for idx in range(adata_t.n_obs):
            row = {
                "timepoint": time_label,
                "samples": float(label_to_model_time_map[time_label]),
                "dataset_kind": dataset_kind,
                "celltype_prediction": str(obs.iloc[idx]["celltype_prediction"]),
                "obs_name": str(adata_t.obs_names[idx]),
            }
            if "region" in obs.columns:
                row["region"] = str(obs.iloc[idx]["region"])
            for feat_idx, feat_name in enumerate(feature_cols):
                row[feat_name] = float(X[idx, feat_idx])
            rows.append(row)
    return pd.DataFrame(rows)


def compute_growth_df(
    adata_dict: dict[str, ad.AnnData],
    label_to_model_time_map: dict[str, float],
    model,
    device: str,
) -> pd.DataFrame:
    frames = []
    for time_label in time_keys:
        adata_t = adata_dict[time_label]
        X = np.asarray(adata_t.X, dtype=np.float32)
        t_value = float(label_to_model_time_map[time_label])
        t_tensor = torch.full((X.shape[0], 1), t_value, dtype=torch.float32, device=device)
        x_tensor = torch.tensor(X, dtype=torch.float32, device=device)
        with torch.no_grad():
            g_values = model.predict_growth(t=t_tensor, x=x_tensor).detach().cpu().numpy().reshape(-1)
        frame = pd.DataFrame(X, columns=[f"x{i}" for i in range(1, X.shape[1] + 1)])
        frame["timepoint"] = time_label
        frame["samples"] = t_value
        frame["dataset_kind"] = "observed" if bool(adata_t.uns.get("is_observed", False)) else "interpolated"
        frame["celltype_prediction"] = adata_t.obs["celltype_prediction"].astype(str).map(canonical_label).values
        frame["g_value"] = g_values
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def plot_daily_g_value_celltype_colored(
    growth_df: pd.DataFrame,
    label_to_color: dict[str, str],
    save_path: Path,
):
    panels = []
    for key in sorted(growth_df["timepoint"].astype(str).unique(), key=lambda x: float(x.replace("D", ""))):
        subset = growth_df[growth_df["timepoint"].astype(str) == key].copy()
        if subset.empty:
            continue
        panels.append(
            {
                "label": str(key),
                "t": float(subset["samples"].iloc[0]),
                "coords": subset[["x1", "x2"]].to_numpy(dtype=np.float32),
                "celltype": subset["celltype_prediction"].astype(str).map(canonical_label).to_numpy(),
                "dataset_kind": subset["dataset_kind"].astype(str).to_numpy(),
                "g": subset["g_value"].to_numpy(dtype=float),
            }
        )

    all_g_concat = np.concatenate([panel["g"] for panel in panels], axis=0)
    g_min_true = float(np.min(all_g_concat))
    g_max_true = float(np.max(all_g_concat))
    g_p5 = float(np.percentile(all_g_concat, 5))
    g_p95 = float(np.percentile(all_g_concat, 95))
    denom = (g_max_true - g_min_true) + 1e-8

    def g_to_norm(g_arr):
        return np.clip((g_arr - g_min_true) / denom, 0, 1)

    coords_concat = np.vstack([panel["coords"] for panel in panels])
    c_min = coords_concat.min(axis=0)
    c_max = coords_concat.max(axis=0)
    c_scale = float((c_max - c_min).max() + 1e-12)

    ncols = min(11, len(panels))
    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 2.6), dpi=300)
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = np.array([axes])
    elif ncols == 1:
        axes = np.array([[ax] for ax in axes])

    for idx, panel in enumerate(panels):
        r = idx // ncols
        c = idx % ncols
        ax = axes[r, c]
        x_plot = (panel["coords"][:, 0] - c_min[0]) / c_scale
        y_plot = (panel["coords"][:, 1] - c_min[1]) / c_scale
        g_norm = g_to_norm(panel["g"])
        alphas = 0.4 + g_norm * (1.0 - 0.4)
        sizes = 0.8 + g_norm * (18.0 - 0.8)
        cts = np.asarray(panel["celltype"], dtype=str)
        for ct in np.unique(cts):
            mask = cts == ct
            ax.scatter(
                x_plot[mask],
                y_plot[mask],
                c=label_to_color.get(ct, "#B0B0B0"),
                s=sizes[mask],
                alpha=alphas[mask],
                linewidths=0,
                rasterized=False,
            )
        tag_values = set(panel["dataset_kind"].tolist())
        tag = "interp" if tag_values == {"interpolated"} else ("orig" if tag_values == {"observed"} else "mixed")
        ax.set_title(f"{panel['label']} ({tag})", fontsize=8, fontweight="bold", pad=2)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal")
        ax.axis("off")

    for j in range(len(panels), nrows * ncols):
        axes[j // ncols, j % ncols].axis("off")

    plt.subplots_adjust(left=0.015, right=0.78, top=0.96, bottom=0.04, wspace=0, hspace=0)

    import matplotlib.lines as mlines

    g_levels_mid = np.linspace(g_p5, g_p95, 8).astype(float)
    g_levels = np.concatenate(([g_min_true], g_levels_mid, [g_max_true]))
    g_norm_levels = g_to_norm(g_levels)
    size_levels = 0.8 + g_norm_levels * (18.0 - 0.8)
    alpha_levels = 0.4 + g_norm_levels * (1.0 - 0.4)
    handles = [
        mlines.Line2D(
            [],
            [],
            color="black",
            marker="o",
            linestyle="None",
            markersize=np.sqrt(max(size, 1.0)),
            alpha=float(alpha),
            label=f"{val:.2f}",
        )
        for val, size, alpha in zip(g_levels[::-1], size_levels[::-1], alpha_levels[::-1])
    ]
    legend = fig.legend(
        handles=handles,
        title="g value",
        loc="center left",
        bbox_to_anchor=(0.81, 0.54),
        frameon=False,
        handletextpad=0.8,
        labelspacing=1.05,
        borderaxespad=0.0,
        fontsize=8,
        title_fontsize=9,
    )
    legend._legend_box.align = "left"

    ct_handles = [
        mpatches.Patch(color=label_to_color[label], label=label)
        for label in sorted(label_to_color)
        if label in set(growth_df["celltype_prediction"].astype(str))
    ]
    fig.legend(
        handles=ct_handles,
        title="Cell type",
        loc="center left",
        bbox_to_anchor=(0.81, 0.08),
        frameon=False,
        fontsize=8,
        title_fontsize=9,
        labelspacing=0.6,
        borderaxespad=0.0,
        handlelength=1.0,
        handletextpad=0.5,
    )

    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return save_path


growth_df = compute_growth_df(daily_adata_dict, label_to_model_time, model, DEVICE)
g_csv_path = GROWTH_DIR / "g_df_with_values_for_heatmap_v2_horizontal.csv"
growth_df.to_csv(g_csv_path, index=False)

g_scatter_path = GROWTH_DIR / "g_value_celltype_colored.svg"
plot_daily_g_value_celltype_colored(growth_df, label_to_color, g_scatter_path)

pivot = _prepare_g_heatmap_matrix(
    df_with_g=growth_df,
    time_col="timepoint",
    label_col="celltype_prediction",
    agg="median",
)

g_heatmap_svg = GROWTH_DIR / "g_heatmap_by_celltype_time_v2_horizontal.svg"
g_heatmap_png = GROWTH_DIR / "g_heatmap_by_celltype_time_v2_horizontal.png"
fig, ax = plt.subplots(figsize=(16, 7), dpi=300)
im = ax.imshow(pivot.values, aspect="auto")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.iloc[i, j]
        if pd.notna(val):
            text_color = "white" if float(val) < 2.0 else "black"
            ax.text(
                j,
                i,
                f"{float(val):.2f}",
                ha="center",
                va="center",
                fontsize=16,
                fontweight="bold",
                color=text_color,
            )
ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=0)
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("Timepoint")
ax.set_ylabel("Cell type")
ax.set_title("Median growth rate (g) by cell type and timepoint")
cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label("g value")
fig.tight_layout()
fig.savefig(g_heatmap_svg, dpi=300, bbox_inches="tight")
fig.savefig(g_heatmap_png, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved g scatter to: {g_scatter_path}")
print(f"Saved g heatmap svg to: {g_heatmap_svg}")
print(f"Saved g heatmap png to: {g_heatmap_png}")
display(SVG(filename=str(g_scatter_path)))
display(SVG(filename=str(g_heatmap_svg)))

In [ ]:
batch_names = ["D4", "D7", "D10", "D14"]


def scale_spatial_coords(ad):
    coords = ad.obsm["spatial"].copy()
    cx = coords[:, 0].mean()
    coords[:, 0] = coords[:, 0] - cx
    ad.obsm["spatial"] = coords / 5000
    return ad


def keep_top_right_cluster(ad, eps=0.06, min_samples=5, prefer="x+y"):
    coords = ad.obsm["spatial"]
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(coords)
    valid = labels >= 0
    if valid.sum() == 0:
        return ad

    labels = labels[valid]
    pts = coords[valid]
    centroids = {lb: pts[labels == lb].mean(axis=0) for lb in np.unique(labels)}

    if prefer == "y":
        best = max(centroids.items(), key=lambda kv: kv[1][1])[0]
    elif prefer == "x":
        best = max(centroids.items(), key=lambda kv: kv[1][0])[0]
    else:
        best = max(centroids.items(), key=lambda kv: kv[1][0] + kv[1][1])[0]

    keep_mask_full = np.zeros(ad.n_obs, dtype=bool)
    keep_mask_full[np.where(valid)[0][labels == best]] = True
    return ad[keep_mask_full].copy()


def _pick_corner(labels, xy, corner="bottom-left"):
    valid = labels >= 0
    lbl = labels[valid]
    pts = xy[valid]
    if pts.shape[0] == 0:
        return np.ones(xy.shape[0], dtype=bool)

    centroids = {k: pts[lbl == k].mean(axis=0) for k in np.unique(lbl)}
    if corner == "top-right":
        score = {k: +(c[0] + c[1]) for k, c in centroids.items()}
    elif corner == "top-left":
        score = {k: +(-c[0] + c[1]) for k, c in centroids.items()}
    elif corner == "bottom-right":
        score = {k: +(c[0] - c[1]) for k, c in centroids.items()}
    elif corner == "bottom-left":
        score = {k: +(-c[0] - c[1]) for k, c in centroids.items()}
    else:
        raise ValueError("corner must be one of: top-right/top-left/bottom-right/bottom-left")
    best = max(score, key=score.get)

    keep_mask_full = np.zeros(xy.shape[0], dtype=bool)
    keep_mask_full[np.where(valid)[0][lbl == best]] = True
    return keep_mask_full


def keep_corner_auto(ad, corner="bottom-left", eps_grid=(0.05, 0.048, 0.046, 0.044, 0.042, 0.040), min_samples=5, k_default=4, k_range=(3, 6)):
    xy = ad.obsm["spatial"]
    best_labels = None
    best_n = -1
    for eps in eps_grid:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(xy)
        n = len(np.unique(labels[labels >= 0]))
        if n > 0 and (best_n == -1 or abs(n - 4) < abs(best_n - 4)):
            best_labels, best_n = labels, n
        if n == 4:
            break

    if best_labels is not None and 3 <= best_n <= 6:
        mask = _pick_corner(best_labels, xy, corner)
        return ad[mask].copy()

    best_k, best_score = None, -1
    for k in range(k_range[0], k_range[1] + 1):
        km = KMeans(n_clusters=k, n_init="auto", random_state=0)
        labels = km.fit_predict(xy)
        try:
            s = silhouette_score(xy, labels) if k > 1 else -1
        except Exception:
            s = -1
        if s > best_score:
            best_score, best_k = s, k

    if best_k is None:
        best_k = k_default

    km = KMeans(n_clusters=best_k, n_init="auto", random_state=0)
    labels = km.fit_predict(xy)
    mask = _pick_corner(labels, xy, corner)
    return ad[mask].copy()


def _pick_extreme(labels, xy, extreme="right"):
    valid = labels >= 0
    lbl = labels[valid]
    pts = xy[valid]
    if pts.shape[0] == 0:
        return np.ones(xy.shape[0], dtype=bool)

    cents = {k: pts[lbl == k].mean(axis=0) for k in np.unique(lbl)}
    if extreme == "right":
        best = max(cents.items(), key=lambda kv: kv[1][0])[0]
    elif extreme == "left":
        best = min(cents.items(), key=lambda kv: kv[1][0])[0]
    elif extreme == "top":
        best = max(cents.items(), key=lambda kv: kv[1][1])[0]
    elif extreme == "bottom":
        best = min(cents.items(), key=lambda kv: kv[1][1])[0]
    else:
        raise ValueError("extreme must be one of: right/left/top/bottom")

    keep = np.zeros(xy.shape[0], dtype=bool)
    keep[np.where(valid)[0][lbl == best]] = True
    return keep


def keep_extreme_cluster(ad, extreme="right", eps_grid=(0.05, 0.048, 0.046, 0.044, 0.042, 0.040), min_samples=5, k_default=2, k_range=(2, 6)):
    xy = ad.obsm["spatial"]
    best_labels, best_n = None, -1
    for eps in eps_grid:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(xy)
        n = len(np.unique(labels[labels >= 0]))
        if n > 0 and (best_n == -1 or abs(n - 2) < abs(best_n - 2)):
            best_labels, best_n = labels, n
        if n >= 2:
            break

    if best_labels is not None and best_n >= 2:
        mask = _pick_extreme(best_labels, xy, extreme)
        return ad[mask].copy()

    best_k, best_score = None, -1
    for k in range(k_range[0], k_range[1] + 1):
        try:
            km = KMeans(n_clusters=k, n_init="auto", random_state=0)
            lbl = km.fit_predict(xy)
            s = silhouette_score(xy, lbl) if k > 1 else -1
        except Exception:
            s = -1
        if s > best_score:
            best_score, best_k = s, k

    if best_k is None:
        best_k = k_default

    km = KMeans(n_clusters=best_k, n_init="auto", random_state=0)
    labels = km.fit_predict(xy)
    mask = _pick_extreme(labels, xy, extreme)
    return ad[mask].copy()


print("Reading merged_with_meta...")
adata = sc.read_h5ad(MERGED_WITH_META_PATH)
adata.obs["timepoint"] = adata.obs["timepoint"].astype("category")
adata.obs["timepoint"] = adata.obs["timepoint"].cat.set_categories(batch_names, ordered=True)

adata_dict = {b: adata[adata.obs["timepoint"] == b].copy() for b in batch_names}
adata_dict_raw = {b: adata[adata.obs["timepoint"] == b].copy() for b in batch_names}

for b in batch_names:
    ad_b_raw = adata_dict_raw[b].copy()
    ad_b = adata_dict[b].copy()
    ad_b = scale_spatial_coords(ad_b)

    if b == "D4":
        ad_b = keep_top_right_cluster(ad_b, eps=0.06, min_samples=5, prefer="x+y")
    elif b == "D7":
        ad_b = keep_corner_auto(
            ad_b,
            corner="bottom-left",
            eps_grid=(0.05, 0.048, 0.046, 0.044, 0.042, 0.040),
            min_samples=5,
            k_default=4,
            k_range=(3, 6),
        )
        y = ad_b.obsm["spatial"][:, 1]
        y_flipped = y.max() - (y - y.min())
        ad_b.obsm["spatial"][:, 1] = y_flipped
    elif b == "D10":
        ad_b = keep_extreme_cluster(
            ad_b,
            extreme="right",
            eps_grid=(0.05, 0.048, 0.046, 0.044, 0.042, 0.040),
            min_samples=5,
            k_default=2,
            k_range=(2, 6),
        )

    adata_dict[b] = ad_b
    adata_dict_raw[b] = ad_b_raw[ad_b.obs_names].copy()

X_orig_list = [np.asarray(adata_dict_raw[b].obsm["spatial"], dtype=float) for b in batch_names]

adata_aligned = sc.read_h5ad(ALIGNED_H5AD_PATH)
adata_aligned.obs["timepoint"] = adata_aligned.obs["timepoint"].astype("category")
adata_aligned.obs["timepoint"] = adata_aligned.obs["timepoint"].cat.set_categories(batch_names, ordered=True)
aligned_basis = "spatial_aligned" if "spatial_aligned" in adata_aligned.obsm else "spatial"
X_aligned_list = []
for b in batch_names:
    subset = adata_aligned[adata_aligned.obs["timepoint"] == b].copy()
    target_obs_names = adata_dict[b].obs_names.intersection(subset.obs_names)
    subset = subset[target_obs_names].copy() if len(target_obs_names) > 0 else subset
    X_aligned_list.append(np.asarray(subset.obsm[aligned_basis], dtype=float))

orig_vs_aligned_path = ALIGN_DIR / "orig_vs_aligned_coordinates.svg"
fig, axes = plt.subplots(2, 1, figsize=(6, 8), dpi=100)
panel_data = [
    ("Original Coordinates", X_orig_list),
    ("Transformed Coordinates", X_aligned_list),
]

for ax, (title_str, data_list) in zip(axes, panel_data):
    all_x = np.concatenate([x[:, 0] for x in data_list])
    all_y = np.concatenate([x[:, 1] for x in data_list])
    x_min, x_max = float(all_x.min()), float(all_x.max())
    y_min, y_max = float(all_y.min()), float(all_y.max())
    for batch_name, color, data in zip(batch_names[::-1], MODERN_TIME_COLORS[::-1], data_list[::-1]):
        ax.scatter(
            data[:, 0],
            data[:, 1],
            c=[color],
            s=6,
            alpha=0.55,
            edgecolors="none",
            label=f"({batch_name})",
        )
    ax.set_title(title_str, fontsize=14, fontweight="bold", pad=15)
    ax.set_xticks([x_min, x_max])
    ax.set_yticks([y_min, y_max])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#333333")
        spine.set_linewidth(0.8)
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        markerscale=3.0,
        frameon=False,
        fontsize=11,
    )
    ax.set_xlabel("X Coordinate", fontsize=11)
    ax.set_ylabel("Y Coordinate", fontsize=11)
    ax.set_aspect("equal")

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.savefig(orig_vs_aligned_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved orig-vs-aligned figure to: {orig_vs_aligned_path}")
display(SVG(filename=str(orig_vs_aligned_path)))

In [ ]:
def run_velocity_strength_diagnostics_formal(
    *,
    df,
    time_keys,
    model,
    annotation_col,
    interaction_threshold,
    velocity_keys=("full", "drift", "interaction"),
    mode="gene",
    time_label_map=None,
    save_dir=None,
    save_suffix="",
    device="cpu",
    point_size=240,
    point_alpha=0.85,
    stream_density=2.9,
    stream_linewidth=0.46,
    n_neighbors=20,
    save_metrics=True,
    linewidth_strength_source="raw",
    linewidth_reference_key="drift",
    linewidth_min_factor=0.05,
    linewidth_max_factor=1.45,
    linewidth_power=1.05,
    stream_grid_size=140,
    color_strength_source="projected",
    stream_color_map="magma_soft",
    color_percentile=90,
    color_floor_quantile=0.05,
    local_linewidth_min_scale=0.90,
    local_linewidth_max_scale=1.35,
    local_linewidth_power=1.0,
    show_stream_colorbar=True,
    streamline_base_color="#000000",
    streamline_alpha=1.0,
    overlay_quiver=False,
    quiver_step=3,
    quiver_width=0.0105,
    quiver_scale=1.0,
    quiver_alpha=1.0,
    quiver_min_speed_quantile=0.08,
    quiver_length_base=0.030,
    quiver_length_boost=0.055,
    show_linewidth_factor_in_title=True,
):
    user_color_map = dict(LEGACY_REGION_COLOR_MAP)
    display_labels = {k: k for k in user_color_map}
    region_colors_map = dict(user_color_map)

    if time_label_map is None:
        time_label_map = {t: f"t = {t}" for t in time_keys}

    def _component_strength(v):
        mag = np.linalg.norm(v, axis=1)
        if mag.size == 0:
            return 0.0
        return float(np.nanpercentile(mag, 90))

    def _cosine_per_cell(a, b, eps=1e-8):
        numerator = np.sum(a * b, axis=1)
        denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + eps
        return numerator / denominator

    def _safe_summary(x):
        x = np.asarray(x, dtype=float)
        if x.size == 0:
            return {"mean": np.nan, "median": np.nan, "p75": np.nan, "p90": np.nan, "p95": np.nan, "max": np.nan}
        return {
            "mean": float(np.nanmean(x)),
            "median": float(np.nanmedian(x)),
            "p75": float(np.nanpercentile(x, 75)),
            "p90": float(np.nanpercentile(x, 90)),
            "p95": float(np.nanpercentile(x, 95)),
            "max": float(np.nanmax(x)),
        }

    def _get_stream_cmap(name):
        if name == "gray_black":
            return mcolors.LinearSegmentedColormap.from_list("gray_black", ["#b8b8b8", "#000000"])
        if name == "magma_soft":
            base = plt.get_cmap("magma")
            return mcolors.LinearSegmentedColormap.from_list("magma_soft", base(np.linspace(0.18, 0.82, 256)))
        return plt.get_cmap(name)

    def _select_quiver_points(coords, uv, color_values, min_speed_quantile=0.15, step=5):
        coords = np.asarray(coords, dtype=float)
        uv = np.asarray(uv, dtype=float)
        color_values = np.asarray(color_values, dtype=float)
        mag = np.linalg.norm(uv, axis=1)
        keep = np.isfinite(coords).all(axis=1) & np.isfinite(uv).all(axis=1) & np.isfinite(color_values) & np.isfinite(mag)
        if not np.any(keep):
            return np.array([], dtype=int)
        valid_idx = np.where(keep)[0]
        mag_valid = mag[keep]
        cut = float(np.nanquantile(mag_valid, min_speed_quantile))
        valid_idx = valid_idx[mag_valid >= cut]
        if valid_idx.size == 0:
            return np.array([], dtype=int)
        coords_valid = coords[valid_idx]
        mag_valid = mag[valid_idx]
        x_min, y_min = coords_valid.min(axis=0)
        x_max, y_max = coords_valid.max(axis=0)
        bins_x = max(10, int(round(stream_grid_size / max(step, 1))))
        bins_y = max(12, int(round(bins_x * (y_max - y_min) / max((x_max - x_min), 1e-8))))
        x_bin = np.clip(((coords_valid[:, 0] - x_min) / max((x_max - x_min), 1e-8) * (bins_x - 1)).astype(int), 0, bins_x - 1)
        y_bin = np.clip(((coords_valid[:, 1] - y_min) / max((y_max - y_min), 1e-8) * (bins_y - 1)).astype(int), 0, bins_y - 1)
        chosen = {}
        for local_i, global_i in enumerate(valid_idx):
            key = (x_bin[local_i], y_bin[local_i])
            if key not in chosen or mag_valid[local_i] > chosen[key][1]:
                chosen[key] = (global_i, mag_valid[local_i])
        return np.array(sorted(v[0] for v in chosen.values()), dtype=int)

    def _plot_region_stream(ax, *, ad_plot, coords, uv, color_values, linewidth_base, density, arrowsize, point_size, point_alpha, color_norm, region_colors_map, stream_color_map, show_stream_colorbar, overlay_quiver, quiver_step, quiver_width, quiver_scale, quiver_alpha, quiver_min_speed_quantile, quiver_length_base, quiver_length_boost):
        cats = ad_plot.obs["region"].cat.categories
        ad_plot.uns["region_colors"] = [region_colors_map.get(c, "#888888") for c in cats]
        scv.pl.velocity_embedding_stream(
            ad_plot,
            basis="spatial",
            color="region",
            ax=ax,
            show=False,
            legend_loc="none",
            size=point_size,
            density=density,
            alpha=point_alpha,
            linewidth=linewidth_base,
            arrowsize=arrowsize,
            min_mass=0.2,
            smooth=0.8 if mode == "gene" else 0.5,
            frameon=False,
            title="",
        )
        for coll in ax.collections:
            if isinstance(coll, LineCollection):
                coll.set_color(streamline_base_color)
                coll.set_alpha(streamline_alpha)
        for patch in ax.patches:
            if isinstance(patch, FancyArrowPatch):
                patch.set_color(streamline_base_color)
                patch.set_alpha(streamline_alpha)
        cmap = _get_stream_cmap(stream_color_map)
        if overlay_quiver:
            selected = _select_quiver_points(coords, uv, color_values, min_speed_quantile=quiver_min_speed_quantile, step=quiver_step)
            if selected.size > 0:
                uv_sel = uv[selected]
                mag_sel = np.linalg.norm(uv_sel, axis=1)
                dir_sel = uv_sel / np.maximum(mag_sel[:, None], 1e-8)
                color_clipped = np.clip(color_values[selected], color_norm.vmin, color_norm.vmax)
                color_scaled = (color_clipped - color_norm.vmin) / max((color_norm.vmax - color_norm.vmin), 1e-8)
                arrow_len = quiver_length_base + quiver_length_boost * color_scaled
                qu_u = dir_sel[:, 0] * arrow_len
                qu_v = dir_sel[:, 1] * arrow_len
                ax.quiver(
                    coords[selected, 0],
                    coords[selected, 1],
                    qu_u,
                    qu_v,
                    color_clipped,
                    cmap=cmap,
                    norm=color_norm,
                    angles="xy",
                    scale_units="xy",
                    scale=quiver_scale,
                    width=quiver_width,
                    headwidth=7.0,
                    headlength=9.0,
                    headaxislength=7.6,
                    minshaft=2.4,
                    minlength=0.0,
                    pivot="tail",
                    alpha=quiver_alpha,
                    zorder=8,
                )
        if show_stream_colorbar:
            sm = plt.cm.ScalarMappable(norm=color_norm, cmap=cmap)
            sm.set_array([])
            plt.colorbar(sm, ax=ax, fraction=0.04, pad=0.01)
        ax.set_axis_off()
        return ax

    data_cache = {}
    print("Pre-computing formal velocity fields...")
    for t in time_keys:
        subset = df[np.isclose(df["samples"].astype(float).values, float(t))].copy()
        if subset.empty:
            continue
        x_cols = sorted([c for c in subset.columns if c.startswith("x") and c[1:].isdigit()], key=lambda c: int(c[1:]))
        x_data = subset[x_cols].to_numpy(dtype=np.float32)
        vel_res = cb.tl.compute_velocity_components(
            data=x_data,
            time_value=float(t),
            model=model,
            interaction_m=1024,
            interaction_threshold=interaction_threshold,
            device=device,
            spatial_dim=2,
        )
        ad_t = ad.AnnData(X=x_data)
        ad_t.obsm["X_spatial"] = x_data[:, :2]
        ad_t.layers["Ms"] = x_data
        ad_t.obs["region"] = subset[annotation_col].astype(str).str.replace("\n", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip().values
        ad_t.obs["region"] = ad_t.obs["region"].astype("category")
        data_cache[t] = (ad_t, vel_res)

    saved_paths = []
    metrics_rows = []
    per_cell_frames = []

    for t in time_keys:
        if t not in data_cache:
            continue
        time_label = time_label_map.get(t, f"t = {t}")
        ad_base, vel_dict = data_cache[t]
        prepared = {}

        for v_key in velocity_keys:
            if v_key not in vel_dict:
                continue
            velocity_high_dim = np.asarray(vel_dict[v_key], dtype=np.float32)
            if mode == "gene":
                gene_data = ad_base.X[:, 2:]
                velocity_gene = velocity_high_dim[:, 2:]
                ad_plot = ad.AnnData(X=gene_data)
                ad_plot.obsm["X_spatial"] = ad_base.obsm["X_spatial"]
                ad_plot.layers["Ms"] = gene_data.copy()
                ad_plot.layers["velocity"] = velocity_gene.copy()
                ad_plot.obs["region"] = ad_base.obs["region"].values
                ad_plot.obs["region"] = ad_plot.obs["region"].astype("category")
                # Formal native gene-space rendering builds the
                # velocity graph in model state space (X), then
                # projects the resulting field back to observed
                # spatial coordinates.
                sc.pp.neighbors(ad_plot, n_neighbors=n_neighbors, use_rep="X")
                scv.tl.velocity_graph(ad_plot, vkey="velocity", xkey="Ms", n_jobs=1)
                scv.tl.velocity_embedding(ad_plot, basis="spatial", vkey="velocity")
                uv = np.asarray(ad_plot.obsm["velocity_spatial"], dtype=float)
                raw_v = np.asarray(velocity_gene, dtype=float)
            else:
                ad_plot = ad_base.copy()
                ad_plot.layers["velocity"] = velocity_high_dim.copy()
                ad_plot.obsm["velocity_spatial"] = velocity_high_dim[:, :2].copy()
                ad_plot.obs["region"] = ad_plot.obs["region"].astype("category")
                uv = np.asarray(ad_plot.obsm["velocity_spatial"], dtype=float)
                raw_v = np.asarray(velocity_high_dim[:, :2], dtype=float)

            mag_projected = np.linalg.norm(uv, axis=1)
            mag_raw = np.linalg.norm(raw_v, axis=1)
            prepared[v_key] = {
                "ad_plot": ad_plot,
                "uv": uv,
                "raw_v": raw_v,
                "mag": mag_projected,
                "mag_raw": mag_raw,
                "strength": _component_strength(raw_v if linewidth_strength_source == "raw" else uv),
                "raw_strength": _component_strength(raw_v),
                "projected_strength": _component_strength(uv),
                "color_values": mag_raw if color_strength_source == "raw" else mag_projected,
            }

        valid_keys = [v_key for v_key in velocity_keys if v_key in prepared]
        eps = 1e-8
        ref_strength = prepared[linewidth_reference_key]["strength"] if linewidth_reference_key in prepared else np.nanmedian([prepared[k]["strength"] for k in valid_keys])
        shared_color_ref = np.nanmax([np.nanpercentile(prepared[k]["color_values"], color_percentile) for k in valid_keys])
        shared_color_floor = np.nanmin([np.nanquantile(prepared[k]["color_values"], color_floor_quantile) for k in valid_keys])
        shared_color_ref = max(shared_color_ref, eps)
        shared_color_floor = min(shared_color_floor, shared_color_ref - eps)
        color_norm = mcolors.Normalize(vmin=shared_color_floor, vmax=shared_color_ref)
        if not np.isfinite(ref_strength) or ref_strength <= eps:
            ref_strength = 1.0

        for v_key in valid_keys:
            ratio = prepared[v_key]["strength"] / (ref_strength + eps)
            lw_factor = float(np.clip(ratio ** linewidth_power, linewidth_min_factor, linewidth_max_factor))
            prepared[v_key]["linewidth_factor"] = lw_factor
            prepared[v_key]["plot_linewidth"] = stream_linewidth * lw_factor
            prepared[v_key]["plot_density"] = stream_density
            prepared[v_key]["plot_arrowsize"] = 0.65
            prepared[v_key]["color_pctl_ref"] = float(np.nanpercentile(prepared[v_key]["color_values"], color_percentile))

        metric_row = {
            "time_model": float(t),
            "time_label": str(time_label),
            "mode": str(mode),
        }
        for key in ["full", "drift", "interaction"]:
            if key in prepared:
                summary = _safe_summary(prepared[key]["mag"])
                for stat_name, stat_value in summary.items():
                    metric_row[f"{key}_{stat_name}_projected_magnitude"] = stat_value
                metric_row[f"{key}_linewidth_factor"] = prepared[key]["linewidth_factor"]
        metrics_rows.append(metric_row)

        region_values = ad_base.obs["region"].astype(str).to_numpy()
        coords = np.asarray(ad_base.obsm["X_spatial"], dtype=float)
        per_cell_df = pd.DataFrame({
            "time_model": float(t),
            "time_label": str(time_label),
            "region": region_values,
            "x": coords[:, 0],
            "y": coords[:, 1],
            "full_mag_projected": prepared["full"]["mag"],
            "drift_mag_projected": prepared["drift"]["mag"],
            "interaction_mag_projected": prepared["interaction"]["mag"],
            "full_mag_raw": prepared["full"]["mag_raw"],
            "drift_mag_raw": prepared["drift"]["mag_raw"],
            "interaction_mag_raw": prepared["interaction"]["mag_raw"],
        })
        per_cell_df["interaction_to_drift_projected_ratio"] = per_cell_df["interaction_mag_projected"] / (per_cell_df["drift_mag_projected"] + eps)
        per_cell_df["cos_interaction_drift_projected"] = _cosine_per_cell(prepared["interaction"]["uv"], prepared["drift"]["uv"])
        per_cell_frames.append(per_cell_df)

        fig, axes = plt.subplots(1, len(velocity_keys), figsize=(4.8 * len(velocity_keys), 6.0))
        if len(velocity_keys) == 1:
            axes = [axes]
        present_cats = set()
        for i, v_key in enumerate(velocity_keys):
            ax = axes[i]
            info = prepared[v_key]
            ad_plot = info["ad_plot"]
            cats = ad_plot.obs["region"].cat.categories
            present_cats.update(cats)
            title_text = f"{v_key.capitalize()}\nraw p90 = {info['raw_strength']:.4f}\nlw x {info['linewidth_factor']:.2f}" if show_linewidth_factor_in_title else v_key.capitalize()
            _plot_region_stream(
                ax,
                ad_plot=ad_plot,
                coords=coords,
                uv=info["uv"],
                color_values=info["color_values"],
                linewidth_base=info["plot_linewidth"],
                density=info["plot_density"],
                arrowsize=info["plot_arrowsize"],
                point_size=point_size,
                point_alpha=point_alpha,
                color_norm=color_norm,
                region_colors_map=region_colors_map,
                stream_color_map=stream_color_map,
                show_stream_colorbar=show_stream_colorbar,
                overlay_quiver=overlay_quiver,
                quiver_step=quiver_step,
                quiver_width=quiver_width,
                quiver_scale=quiver_scale,
                quiver_alpha=quiver_alpha,
                quiver_min_speed_quantile=quiver_min_speed_quantile,
                quiver_length_base=quiver_length_base,
                quiver_length_boost=quiver_length_boost,
            )
            ax.set_title(title_text)
        legend_handles = [mpatches.Patch(color=region_colors_map.get(c, "#888888"), label=display_labels.get(c, c)) for c in sorted(list(present_cats), key=lambda x: list(region_colors_map.keys()).index(x) if x in region_colors_map else 999)]
        fig.legend(handles=legend_handles, title="Regions", loc="center left", bbox_to_anchor=(0.92, 0.5), frameon=False, labelspacing=1.5)
        plt.suptitle(f"Velocity Decomposition at {time_label} ({mode})", fontsize=16)
        plt.tight_layout(rect=[0, 0, 0.9, 0.93])
        if save_dir:
            full_path = os.path.join(save_dir, f"velocity_components_linewidth_{str(time_label).replace(' ', '_')}_{mode}{save_suffix}.svg")
            plt.savefig(full_path, dpi=300, bbox_inches="tight")
            saved_paths.append(full_path)
        plt.show()

        diag_df = per_cell_df.copy()
        fig, axes = plt.subplots(2, 3, figsize=(17, 10), dpi=250)
        sns.histplot(diag_df["full_mag_raw"], color="#222222", label="full", stat="density", bins=40, alpha=0.40, ax=axes[0, 0])
        sns.histplot(diag_df["drift_mag_raw"], color="#4c78a8", label="drift", stat="density", bins=40, alpha=0.45, ax=axes[0, 0])
        sns.histplot(diag_df["interaction_mag_raw"], color="#e45756", label="interaction", stat="density", bins=40, alpha=0.45, ax=axes[0, 0])
        axes[0, 0].legend(frameon=False)
        axes[0, 0].set_title("Raw magnitude distribution")

        sns.histplot(diag_df["full_mag_projected"], color="#222222", label="full", stat="density", bins=40, alpha=0.40, ax=axes[0, 1])
        sns.histplot(diag_df["drift_mag_projected"], color="#4c78a8", label="drift", stat="density", bins=40, alpha=0.45, ax=axes[0, 1])
        sns.histplot(diag_df["interaction_mag_projected"], color="#e45756", label="interaction", stat="density", bins=40, alpha=0.45, ax=axes[0, 1])
        axes[0, 1].legend(frameon=False)
        axes[0, 1].set_title("Projected magnitude distribution")

        full_vmax = float(np.nanpercentile(diag_df["full_mag_projected"], 95))
        if not np.isfinite(full_vmax) or full_vmax <= 0:
            full_vmax = float(np.nanmax(diag_df["full_mag_projected"])) if np.isfinite(np.nanmax(diag_df["full_mag_projected"])) else 1.0
        full_vmax = max(full_vmax, 1e-8)
        sc0 = axes[0, 2].scatter(
            diag_df["x"],
            diag_df["y"],
            c=np.clip(diag_df["full_mag_projected"], 0, full_vmax),
            s=18,
            cmap="inferno",
            vmin=0,
            vmax=full_vmax,
        )
        axes[0, 2].set_title("Spatial full velocity magnitude (projected)")
        axes[0, 2].set_aspect("equal", adjustable="box")
        plt.colorbar(sc0, ax=axes[0, 2], fraction=0.046, pad=0.04)

        sc1 = axes[1, 0].scatter(diag_df["x"], diag_df["y"], c=np.clip(diag_df["interaction_to_drift_projected_ratio"], 0, 3), s=18, cmap="viridis")
        axes[1, 0].set_title("Spatial ratio: |interaction| / |drift| (projected)")
        axes[1, 0].set_aspect("equal", adjustable="box")
        plt.colorbar(sc1, ax=axes[1, 0], fraction=0.046, pad=0.04)

        sc2 = axes[1, 1].scatter(diag_df["x"], diag_df["y"], c=diag_df["cos_interaction_drift_projected"], s=18, cmap="coolwarm", vmin=-1, vmax=1)
        axes[1, 1].set_title("Spatial cosine: cos(interaction, drift)")
        axes[1, 1].set_aspect("equal", adjustable="box")
        plt.colorbar(sc2, ax=axes[1, 1], fraction=0.046, pad=0.04)

        axes[1, 2].axis("off")
        axes[1, 2].text(
            0.02,
            0.98,
            "\n".join([
                f"Full raw p90: {prepared['full']['raw_strength']:.4f}",
                f"Full projected p90: {float(np.nanpercentile(diag_df['full_mag_projected'], 90)):.4f}",
                f"Drift raw p90: {prepared['drift']['raw_strength']:.4f}",
                f"Interaction raw p90: {prepared['interaction']['raw_strength']:.4f}",
                f"Median |interaction| / |drift|: {float(np.nanmedian(diag_df['interaction_to_drift_projected_ratio'])):.3f}",
                f"Median cos(interaction, drift): {float(np.nanmedian(diag_df['cos_interaction_drift_projected'])):.3f}",
            ]),
            va="top",
            ha="left",
            fontsize=11,
            family="monospace",
        )
        axes[1, 2].set_title("Strength summary")
        plt.tight_layout()
        if save_dir:
            diag_path = os.path.join(save_dir, f"velocity_component_diagnostics_{str(time_label).replace(' ', '_')}{save_suffix}.svg")
            plt.savefig(diag_path, dpi=300, bbox_inches="tight")
            saved_paths.append(diag_path)
        plt.show()

    per_cell_df_all = pd.concat(per_cell_frames, ignore_index=True)
    metrics_df = pd.DataFrame(metrics_rows)
    if save_dir and save_metrics:
        per_cell_path = os.path.join(save_dir, f"velocity_component_per_cell_diagnostics_{mode}{save_suffix}.csv")
        metrics_path = os.path.join(save_dir, f"velocity_component_linewidth_metrics_{mode}{save_suffix}.csv")
        per_cell_df_all.to_csv(per_cell_path, index=False)
        metrics_df.to_csv(metrics_path, index=False)
        saved_paths.extend([per_cell_path, metrics_path])
        print(f"Saved per-cell diagnostics: {per_cell_path}")
        print(f"Saved metrics: {metrics_path}")

    return {
        "saved_paths": saved_paths,
        "metrics_df": metrics_df,
        "per_cell_df": per_cell_df_all,
    }


observed_df = build_daily_feature_dataframe({key: daily_adata_dict[key] for key in observed_time_keys}, label_to_model_time)
if "region" not in observed_df.columns:
    raise KeyError("Observed daily dataframe is missing region for velocity diagnostics.")

SELECT_REAL_DAYS = [10]
REAL_TO_MODEL_TIME = {4.0: 0.0, 7.0: 1.0, 10.0: 2.0, 14.0: 3.0}
TIME_KEYS = [REAL_TO_MODEL_TIME[float(day)] for day in SELECT_REAL_DAYS]
TIME_LABEL_MAP = {REAL_TO_MODEL_TIME[float(day)]: f"D{int(day)}" for day in SELECT_REAL_DAYS}

stderr_buffer = io.StringIO()
with redirect_stderr(stderr_buffer):
    velocity_result = run_velocity_strength_diagnostics_formal(
        df=observed_df,
        time_keys=TIME_KEYS,
        model=model,
        annotation_col="region",
        interaction_threshold=interaction_threshold,
        velocity_keys=("full", "drift", "interaction"),
        mode="gene",
        time_label_map=TIME_LABEL_MAP,
        save_dir=str(VELOCITY_DIR),
        save_suffix="_region",
        device=DEVICE,
    )

print("Saved velocity outputs:")
for p in velocity_result["saved_paths"]:
    print(p)

display(SVG(filename=str(VELOCITY_DIR / "velocity_components_linewidth_D10_gene_region.svg")))
display(SVG(filename=str(VELOCITY_DIR / "velocity_component_diagnostics_D10_region.svg")))

In [ ]:
summary = {
    "input_run_dir": str(INTERP_RUN_DIR),
    "aligned_h5ad": str(ALIGNED_H5AD_PATH),
        "original_h5ad": str(ORIGINAL_H5AD_PATH),
        "merged_with_meta_h5ad": str(MERGED_WITH_META_PATH),
    "model_dir": str(MODEL_DIR),
    "device": DEVICE,
    "time_keys": time_keys,
    "observed_time_keys": observed_time_keys,
    "outputs": {
        "g_heatmap_svg": str(GROWTH_DIR / "g_heatmap_by_celltype_time_v2_horizontal.svg"),
        "g_heatmap_png": str(GROWTH_DIR / "g_heatmap_by_celltype_time_v2_horizontal.png"),
        "g_value_scatter_svg": str(GROWTH_DIR / "g_value_celltype_colored.svg"),
        "g_value_csv": str(GROWTH_DIR / "g_df_with_values_for_heatmap_v2_horizontal.csv"),
        "orig_vs_aligned_svg": str(ALIGN_DIR / "orig_vs_aligned_coordinates.svg"),
        "velocity_component_diagnostics_svg": str(VELOCITY_DIR / "velocity_component_diagnostics_D10_region.svg"),
        "velocity_components_linewidth_svg": str(VELOCITY_DIR / "velocity_components_linewidth_D10_gene_region.svg"),
    },
}

summary_path = OUTPUT_DIR / "supplementary_replot_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved supplementary summary to: {summary_path}")